## Getting started
First, we check the version of Tensorflow and set the seeds for reproducibility.

In [1]:
import tensorflow as tf

import numpy as np
import os
import matplotlib.pyplot as plt
import random
from tqdm import tqdm

seed = 42
img_size = (256,)*2 #gt_image size, and network output size

preprocess_data = True # turn False if you have ready your data
data_augmentation = False
BATCH_SIZE = 1

def set_seed(seedValue=42):
  """Sets the seed on multiple python modules to obtain results as
  reproducible as possible.
  Args:
  seedValue (int, optional): seed value.
  """
  np.random.seed(seed=seedValue)
  tf.random.set_seed(seedValue)
  os.environ["PYTHONHASHSEED"]=str(seedValue)
  random.seed(seedValue)
set_seed(seed)
print("Tensorflow version: ", tf.__version__ )

Tensorflow version:  2.8.0


In [2]:
# https://note.nkmk.me/en/python-numpy-generate-gradation-image/
def get_gradient_2d(start, stop, width, height, is_horizontal):
    if is_horizontal:
        return np.tile(np.linspace(start, stop, width), (height, 1))
    else:
        return np.tile(np.linspace(start, stop, height), (width, 1)).T
def get_gradient_3d(width, height, start_list, stop_list, is_horizontal_list):
    result = np.zeros((height, width, len(start_list)), dtype=np.float64)

    for i, (start, stop, is_horizontal) in enumerate(zip(start_list, stop_list, is_horizontal_list)):
        result[:, :, i] = get_gradient_2d(start, stop, width, height, is_horizontal)

    return result


In [3]:
from PIL import Image

def add_padding(np_img, multiple = 256):
    '''
    Given a numpy array, add padding to the image so that the image is a multiple of 256x256
    
    Args:
      np_img: the image to be padded
    
    Returns:
      A numpy array of the image with the padding added.
    '''

    image = Image.fromarray(np_img)
    height, width, *_ = np_img.shape

    if not width%multiple and not height%multiple:
        return np_img
    
    x = width/multiple
    y = height/multiple

    x = max(x,y)
    y = x

    new_width = int(np.ceil(x))*multiple
    new_height = int(np.ceil(y))*multiple

    left = int( (new_width - width)/2 )
    top = int( (new_height - height)/2 )
    
    if image.mode == 'RGB':
        from_color = tuple(np.random.randint(0, 256, size=3))
        to_color = (0,0,0) if np.random.random() < 0.5 else (255,255,255)
        array = get_gradient_3d(new_width, new_height, from_color, to_color, (True, False, False))

        result = Image.fromarray(np.uint8(array))
        #result = Image.new(image.mode, (new_width, new_height), color)
    else: # single channel
        result = Image.new(image.mode, (new_width, new_height), 0)
    result.paste(image, (left, top))

    return np.array(result)

In [4]:
def display(display_list, fig_size = (14,5)):
  plt.figure(figsize=fig_size)

  title = ['Image', 'Mix Image', 'Heatmap Image']

  for i in range(len(display_list)):
    plt.subplot(1, len(display_list), i+1)
    plt.title(title[i])
    plt.imshow(tf.keras.preprocessing.image.array_to_img(display_list[i]))
  plt.show()
  

In [5]:
from skimage.measure import label, regionprops

def get_just_hand(img, mask):
    #img = label(img, connectivity=2)
    props = regionprops(mask) # (min_row, min_col, max_row, max_col) -- [min; max)
    min_row, min_col, max_row, max_col = props[0]['bbox']
    return img[min_row:max_row+2, min_col-2:max_col+2], mask[min_row:max_row+2, min_col-2:max_col+2]


"\nimg = np.array(Image.open(img_paths_label[0][0]))[:,:,0]<100\nplt.figure(dpi=500)\nimg = get_just_hand(img)\nplt.imshow(img, 'gray')\n"

In [6]:
from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img

class Data(keras.utils.Sequence):
    """Helper to iterate over the data (as Numpy arrays)."""

    def __init__(self, batch_size, img_size, input_img_paths, labels, data_augmentation=False):
        self.batch_size = batch_size
        self.img_size = img_size
        self.input_img_paths = input_img_paths
        self.data_augmentation = data_augmentation
        self.labels = labels

    def __len__(self):
        return len(self.input_img_paths) // self.batch_size

    def get_labels(self):
        return self.labels

    def __getitem__(self, idx):
        """Returns tuple (input, target) correspond to batch #idx."""
        i = idx * self.batch_size
        batch_input_img_paths = self.input_img_paths[i : i + self.batch_size]
        batch_input_labels = self.labels[i : i + self.batch_size]

        x = np.zeros((self.batch_size,) + self.img_size + (3,), dtype="float32")
        y = np.zeros((self.batch_size,) + (1,), dtype="float32")

        for j in range(len(batch_input_img_paths)):
            x_path = batch_input_img_paths[j]
            x_image = Image.open(x_path)

            if self.data_augmentation:
                if random.random() < 0.5:
                    x_image = x_image.transpose(method=Image.FLIP_LEFT_RIGHT)
                if random.random() < 0.5:
                    x_image = x_image.transpose(method=Image.FLIP_TOP_BOTTOM)
                if random.random() < 0.5:
                    for _ in range(random.randint(1, 4)):
                        x_image = x_image.transpose(method=Image.ROTATE_90)
            
            print(x_image)
            x[j] = np.array(x_image, dtype=np.float32) / 255 # normalize x
            y[j] = batch_input_labels[j]

            x_image.close()
        return x,y

## Data modifications

### Splits

Prepare paths of input images and target segmentation masks

In [7]:
from tqdm import tqdm
from glob import glob
from sklearn.model_selection import train_test_split

input_dir = "./masks/"

male_mask_paths = glob(input_dir+"Hombres/*")
female_mask_paths = glob(input_dir+"Mujeres/*")

input_dir = "./mask_correspond/"

male_img_paths = glob(input_dir+"Hombres máscaras/*")
female_img_paths = glob(input_dir+"Mujeres máscaras/*")

# remove path and extension, get only base name
clean_m_mask_names = [os.path.splitext(os.path.basename(x))[0] for x in male_mask_paths]
clean_f_mask_names = [os.path.splitext(os.path.basename(x))[0] for x in female_mask_paths]

clean_m_img_names = [os.path.splitext(os.path.basename(x))[0] for x in male_img_paths]
clean_f_img_names = [os.path.splitext(os.path.basename(x))[0] for x in female_img_paths]

# check if image does exist in both folders
_male_names = set(clean_m_mask_names).intersection(set(clean_m_img_names))
_female_names = set(clean_f_mask_names).intersection(set(clean_f_img_names))

print("Errors m: ", set(clean_m_mask_names).union(set(clean_m_img_names))-_male_names)
print("Errors f: ", set(clean_f_mask_names).union(set(clean_f_img_names))-_female_names)

print("pre-cleaning, number of male   (x,y):", len(male_mask_paths), 'img:', len(male_img_paths) )
print("pre-cleaning, number of female (x,y):", len(female_mask_paths), 'img:', len(female_img_paths) )
print('')
print("number of male   (x,y):", len(_male_names))
print("number of female (x,y):", len(_female_names))

Errors m:  {'58.1.M_Juven', '76.1.M_Juven_', '43.1.M_Child', '30.1.M_Child', '67.1.M_Child_', '55.1.M_Juven', '2.5.M_Child', '62.1.M_Juven', '7.0.M_Child', '28.1.M_Child'}
Errors f:  set()
pre-cleaning, number of male   (x,y): 120 img: 120
pre-cleaning, number of female (x,y): 127 img: 127

number of male   (x,y): 115
number of female (x,y): 127


In [8]:
img_paths = list(_male_names) + list(_female_names)
labels = [1]*len(_male_names) + [0]*len(_female_names)

img_paths_label = list(zip(img_paths, labels))

train_img_paths, tmp_img_paths = train_test_split(
    img_paths_label, 
    test_size=0.40,
    stratify= labels,
    random_state=42)

tmp_np = np.asarray(tmp_img_paths)

val_img_paths, test_img_paths = train_test_split( # mitad del 40% (tmp) en cada dataset
    tmp_img_paths, 
    test_size=0.50,
    stratify= list(tmp_np[:,1]),
    random_state=42)

# Balance classes
test_img_paths.append(val_img_paths.pop(0))

total_n_img = len(img_paths)
print("\nNumber of samples (train): \t{}  --  {}%".format( 
    len(train_img_paths), 
    round(( len(train_img_paths) * 100) / total_n_img )))
print("Number of samples (test): \t{}  --  {}%".format( 
    len(test_img_paths), 
    round(( len(test_img_paths) * 100) / total_n_img )))
print("Number of samples (val): \t{}  --  {}%".format( 
    len(val_img_paths), 
    round(( len(val_img_paths) * 100) / total_n_img )))
print("\nMale images: \t",len(male_img_paths))
print("Female images: \t",len(female_img_paths))
print(len(male_img_paths)+len(female_img_paths), "/",len(train_img_paths)+len(test_img_paths)+len(val_img_paths))


Number of samples (train): 	145  --  60%
Number of samples (test): 	50  --  21%
Number of samples (val): 	47  --  19%

Male images: 	 120
Female images: 	 127
247 / 242


### Save modified images

In [10]:
from PIL import Image
import imgaug.augmenters as iaa
Image.MAX_IMAGE_PIXELS = 300000000 # there are large images

aug = iaa.CLAHE()

# data
splits = [train_img_paths, val_img_paths, test_img_paths]

# out directory names
main_Dir= "./Data_segmentation/"
dirNames=[main_Dir + "train/", main_Dir + "val/", main_Dir + "test/"]
data_type = ["x/", "y/"]
errors = []

male_img_dir = os.path.dirname(male_img_paths[0])
male_mask_dir = os.path.dirname(male_mask_paths[0])
female_img_dir = os.path.dirname(female_img_paths[0])
female_mask_dir = os.path.dirname(female_mask_paths[0])

male_img_ext = os.path.splitext(male_img_paths[0])[1]
male_mask_ext = os.path.splitext(male_mask_paths[0])[1]
female_img_ext = os.path.splitext(female_img_paths[0])[1]
female_mask_ext = os.path.splitext(female_mask_paths[0])[1]

if preprocess_data:

    for i, ds in enumerate(splits):
        outdir = dirNames[i]
        
        # create folders if not exist
        for folder in data_type:
            if not os.path.exists(outdir + folder):
                os.makedirs(outdir + folder)

        for image_name, label in tqdm(ds):
            
            # Skip processed images
            if os.path.exists(outdir + data_type[0] + str(label) + "_" + image_name):
                continue
            
            # Open images
            try:
                if image_name.find('Hand') != -1:
                    # Get file names
                    image_name = '/'.join(image_name.split("\\"))
                    label = '/'.join(label.split("\\"))
                    image = Image.open(image_name)
                    mask = Image.open(label)
                    
                    image_name = os.path.splitext(os.path.basename(image_name))[0]
                    label = os.path.splitext(os.path.basename(label))[0]
                else:
                    img_dir = male_img_dir if label else female_img_dir
                    mask_dir = male_mask_dir if label else female_mask_dir
                    img_ext = male_img_ext if label else female_img_ext
                    mask_ext = male_mask_ext if label else female_mask_ext

                    image = Image.open(os.path.join(img_dir, image_name + img_ext))
                    mask = Image.open(os.path.join(mask_dir, image_name + mask_ext))
            except:
                errors.append(image_name)
                continue

            # Convert (RGBA,...) images to RGB
            if image.mode != 'RGB':
                image = image.convert('RGB')

            # labeled img - cropp by bbox
            mask = np.array(mask)
            if len(mask.shape) == 3: 
                mask = mask[:,:,0]
            
            if image_name.find('Hand') != -1:
                mask = (mask > 100).astype(np.uint8)
            else:
                mask = (mask < 100).astype(np.uint8)

            image, mask = get_just_hand(np.array(image), mask)
            image = aug(image=image)

            # Adding a padding of zeros to the image.
            image = add_padding(image)
            mask = add_padding(mask) *255

            mask = np.expand_dims(mask, axis=-1)
            mask = np.concatenate([mask, mask, mask], axis=-1)

            image = Image.fromarray(image.astype(np.uint8))
            mask = Image.fromarray(mask.astype(np.uint8))
            
            # Apply moddifications (crappify, resize, ...)
            gt_img = image.resize(img_size, Image.BILINEAR)
            gt_mask = mask.resize(img_size, Image.NEAREST)

            # Save images (without compression)
            image_name, extension = os.path.splitext(image_name)
            gt_img.save(outdir + data_type[0] + str(label) + "_" + image_name + '.png', quality=100)
            gt_mask.save(outdir + data_type[1] + str(label) + "_" + image_name + '.png', quality=100)
        
    print()
    print("Errors: ", errors)

100%|██████████| 50/50 [00:13<00:00,  3.77it/s]


Errors:  []


## Check data

In [11]:
## directory names
main_Dir= "./Data_mask/"
dirNames=[main_Dir + "train/", main_Dir + "val/", main_Dir + "test/"]
data_type = ["img/"]

from glob import glob
from collections import Counter

# Get paths
x_path = data_type[0]+ "*"

train_x_paths = glob(dirNames[0] + x_path)
#train_x_paths.sort()
train_labels = np.asarray([int(os.path.basename(x)[0]) for x in train_x_paths])

val_x_paths = glob(dirNames[1] + x_path)
#val_x_paths.sort()
val_labels = np.asarray([int(os.path.basename(x)[0]) for x in val_x_paths])

test_x_paths = glob(dirNames[2] + x_path)
#test_x_paths.sort()
test_labels = np.asarray([int(os.path.basename(x)[0]) for x in test_x_paths])

#test_x_paths.pop(0) # make balanced
#test_labels = test_labels[1:]

In [12]:
# data generator
train_dataset = Data( BATCH_SIZE, img_size, train_x_paths, train_labels, data_augmentation)
test_dataset = Data( BATCH_SIZE, img_size, test_x_paths, test_labels)
val_dataset = Data( BATCH_SIZE, img_size, val_x_paths, val_labels)

In [ ]:
print("train:", Counter(train_dataset.get_labels()))
print("test: ", Counter(test_dataset.get_labels()))
print("val:  ", Counter(val_dataset.get_labels()))

In [ ]:
sample_image, sample_label = train_dataset[25]
sample_image, sample_label = sample_image[0], sample_label[0] # first batch image
plt.figure(dpi=50)
plt.imshow(sample_image)
print("label: ", sample_label)